In [1]:
import pyodbc
import pandas as pd

# Connect to SQL Server
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=RetailDW;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

print("Connected successfully")

Connected successfully


In [ ]:
# View 1: Cohort Analysis
cohort_sql = """
CREATE VIEW vw_cohort_analysis AS
WITH tbl_acquisition AS (
    SELECT 
        CustomerID,
        DATEFROMPARTS(YEAR(MIN(InvoiceDate)), MONTH(MIN(InvoiceDate)), 1) AS CohortMonth
    FROM retail_sales
    WHERE CustomerID IS NOT NULL
    GROUP BY CustomerID
),
tbl_activity AS (
    SELECT
        r.CustomerID, 
        DATEFROMPARTS(YEAR(r.InvoiceDate), MONTH(r.InvoiceDate), 1) AS ActivityMonth,
        a.CohortMonth
    FROM retail_sales AS r
    JOIN tbl_acquisition AS a ON r.CustomerID = a.CustomerID
),
tbl_cohort AS (
    SELECT
        CohortMonth,
        ActivityMonth,
        DATEDIFF(MONTH, CohortMonth, ActivityMonth) AS MonthNumber,
        COUNT(DISTINCT CustomerID) AS ActiveCustomers
    FROM tbl_activity
    GROUP BY CohortMonth, ActivityMonth, DATEDIFF(MONTH, CohortMonth, ActivityMonth)
)
SELECT CohortMonth, MonthNumber, ActiveCustomers
FROM tbl_cohort
"""

In [ ]:
# View 2: A/B Test
ab_test_sql = """
CREATE VIEW vw_ab_test_summary AS
SELECT
    TestGroup,
    COUNT(CustomerID) AS NumberOfCustomers,
    ROUND(AVG(TotalRevenue), 2) AS AvgCustomerRevenue,
    ROUND(AVG(AvgOrderValue), 2) AS AvgOrderValue,
    ROUND(AVG(TotalOrders), 2) AS AvgOrders
FROM vw_ab_test
GROUP BY TestGroup
"""

In [ ]:
# View 3: A/B Test Summary  
ab_summary_sql = """
CREATE VIEW vw_ab_test_summary AS
SELECT
    TestGroup,
    COUNT(CustomerID) AS NumberOfCustomers,
    ROUND(AVG(TotalRevenue), 2) AS AvgCustomerRevenue,
    ROUND(AVG(AvgOrderValue), 2) AS AvgOrderValue,
    ROUND(AVG(TotalOrders), 2) AS AvgOrders
FROM vw_ab_test
GROUP BY TestGroup
"""

In [ ]:
print("SQL views documented")

In [6]:
import requests
import json
import pandas as pd

# UK CPI inflation data from Bank of England public API
url = "https://www.bankofengland.co.uk/boeapps/database/fromshowcolumns.asp?Travel=NIxIRxSUx&FromSeries=1&ToSeries=50&DAT=RNG&FD=1&FM=Jan&FY=2010&TD=1&TM=Jan&TY=2012&VFD=Y&html.x=66&html.y=26&C=CPIH&Filter=N&csv.x=yes"

# Alternative — use Open Exchange Rates (no auth needed for base)
# Let's use a simpler guaranteed JSON API
url = "https://restcountries.com/v3.1/alpha/gb"

response = requests.get(url)
print(f"Status code: {response.status_code}")
print(response.text[:500])

Status code: 200
[{"tld":[".uk"],"cca2":"GB","ccn3":"826","cca3":"GBR","cioc":"GBR","independent":true,"status":"officially-assigned","unMember":true,"idd":{"root":"+4","suffixes":["4"]},"capital":["London"],"altSpellings":["GB","UK","Great Britain"],"region":"Europe","subregion":"Northern Europe","landlocked":false,"borders":["IRL"],"area":244376.0,"maps":{"googleMaps":"https://goo.gl/maps/FoDtc3UKMkFsXAjHA","openStreetMaps":"https://www.openstreetmap.org/relation/62149"},"population":69281437,"car":{"signs":["


In [7]:
# Parse the JSON
data = response.json()

# It returns a list — get the first item
country = data[0]

# Extract relevant fields
extracted = {
    'name': country['name']['common'],
    'capital': country['capital'][0],
    'population': country['population'],
    'region': country['region'],
    'subregion': country['subregion'],
    'currency': list(country['currencies'].keys())[0],
    'currency_name': list(country['currencies'].values())[0]['name'],
    'languages': list(country['languages'].values()),
    'area_km2': country['area'],
    'timezones': country['timezones']
}

# Convert to DataFrame
df_country = pd.DataFrame([extracted])
print(df_country)

             name capital  population  region        subregion currency  \
0  United Kingdom  London    69281437  Europe  Northern Europe      GBP   

   currency_name  languages  area_km2  \
0  British pound  [English]  244376.0   

                                           timezones  
0  [UTC-08:00, UTC-05:00, UTC-04:00, UTC-03:00, U...  


In [8]:
# Get all unique countries from our retail database
countries_df = pd.read_sql("""
    SELECT DISTINCT Country FROM retail_sales
    WHERE Country != 'Unspecified'
    ORDER BY Country
""", conn)

print(f"Countries in our dataset: {len(countries_df)}")
print(countries_df['Country'].tolist())

C:\Users\rashi\AppData\Local\Temp\ipykernel_35424\2998446747.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  countries_df = pd.read_sql("""


Countries in our dataset: 37
['Australia', 'Austria', 'Bahrain', 'Belgium', 'Brazil', 'Canada', 'Channel Islands', 'Cyprus', 'Czech Republic', 'Denmark', 'EIRE', 'European Community', 'Finland', 'France', 'Germany', 'Greece', 'Hong Kong', 'Iceland', 'Israel', 'Italy', 'Japan', 'Lebanon', 'Lithuania', 'Malta', 'Netherlands', 'Norway', 'Poland', 'Portugal', 'RSA', 'Saudi Arabia', 'Singapore', 'Spain', 'Sweden', 'Switzerland', 'United Arab Emirates', 'United Kingdom', 'USA']


In [11]:
# Map country names to ISO codes for the API
# Some names in our dataset differ from API names
country_map = {
    'EIRE': 'IE',
    'Czech Republic': 'CZ',
    'Channel Islands': 'JE',
    'European Community': None,  # not a country, skip
    'RSA': 'ZA',
    'USA': 'US',
    'United Kingdom': 'GB'
}

# Pull data for all countries
country_data = []

for country in countries_df['Country'].tolist():
    try:
        # Use name search for most countries
        if country in country_map:
            code = country_map[country]
            if code is None:
                continue
            url = f"https://restcountries.com/v3.1/alpha/{code}"
        else:
            url = f"https://restcountries.com/v3.1/name/{country}?fullText=true"
        
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()[0]
            country_data.append({
                'Country': country,
                'Population': data.get('population', None),
                'Region': data.get('region', None),
                'Subregion': data.get('subregion', None),
                'Area_km2': data.get('area', None)
            })
    except:
        pass

df_enriched = pd.DataFrame(country_data)
print(f"Successfully retrieved: {len(df_enriched)} countries")
df_enriched.head(10)

Successfully retrieved: 36 countries


,Country,Population,Region,Subregion,Area_km2
0,Australia,27536874,Oceania,Australia and New Zealand,7692024.0
1,Austria,9200931,Europe,Central Europe,83871.0
2,Bahrain,1594654,Asia,Western Asia,765.0
3,Belgium,11825551,Europe,Western Europe,30528.0
4,Brazil,213421037,Americas,South America,8515767.0
5,Canada,41651653,Americas,North America,9984670.0
6,Channel Islands,103267,Europe,Northern Europe,116.0
7,Cyprus,1442614,Europe,Southern Europe,9251.0
8,Czech Republic,10882341,Europe,Central Europe,78865.0
9,Denmark,6011488,Europe,Northern Europe,43094.0


In [3]:
from sqlalchemy import create_engine
import urllib

connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=RetailDW;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

connection_url = f"mssql+pyodbc:///?odbc_connect={urllib.parse.quote_plus(connection_string)}"
engine = create_engine(connection_url)

df_enriched.to_sql(
    name='country_enriched',
    con=engine,
    if_exists='replace',
    index=False
)

print(f"Loaded {len(df_enriched)} rows into country_enriched table")

NameError: name 'df_enriched' is not defined

In [2]:
import scipy.stats as stats
import numpy as np

# Pull individual customer revenue for each group
ab_data = pd.read_sql("""
    SELECT TestGroup, AvgOrderValue
    FROM vw_ab_test
    WHERE AvgOrderValue IS NOT NULL
""", conn)

C:\Users\rashi\AppData\Local\Temp\ipykernel_61464\2786501628.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ab_data = pd.read_sql("""


In [4]:
# Split into two groups
control = ab_data[ab_data['TestGroup'] == 'Control']['AvgOrderValue']
treatment = ab_data[ab_data['TestGroup'] == 'Treatment']['AvgOrderValue']

print(f"Control group: {len(control)} customers, mean AOV: £{control.mean():.2f}")
print(f"Treatment group: {len(treatment)} customers, mean AOV: £{treatment.mean():.2f}")
print(f"Difference: £{treatment.mean() - control.mean():.2f}")

Control group: 2172 customers, mean AOV: £98.46
Treatment group: 2166 customers, mean AOV: £38.16
Difference: £-60.30


In [5]:
# Run independent samples t-test
t_stat, p_value = stats.ttest_ind(control, treatment)

print(f"\nt-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

# Interpret the result
alpha = 0.05
if p_value < alpha:
    print(f"\nResult: STATISTICALLY SIGNIFICANT (p={p_value:.4f} < {alpha})")
    print("The difference in average order value between groups is unlikely to be due to chance.")
else:
    print(f"\nResult: NOT STATISTICALLY SIGNIFICANT (p={p_value:.4f} >= {alpha})")
    print("The difference in average order value between groups could be due to chance.")


t-statistic: 1.3529
p-value: 0.1762

Result: NOT STATISTICALLY SIGNIFICANT (p=0.1762 >= 0.05)
The difference in average order value between groups could be due to chance.
